In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Environment Config
# -------------------------------
import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema, get_semantic_layer_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
semantic_layer_schema = get_semantic_layer_schema()
mart_schema = get_mart_schema()  

# -------------------------------
# Create kpi View (Semantic Layer for Analysts)
# -------------------------------
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.{semantic_layer_schema}.customer_kpi AS
WITH ranked_customers AS (
    SELECT
        customer_id,
        customer_name,
        segment,
        region,
        total_sales,
        total_profit,
        total_orders,
        avg_order_value,
        recency_days,
        customer_lifetime_days,
        discount_sensitivity,
        profitability_flag,
        RANK() OVER (PARTITION BY region ORDER BY total_sales DESC) AS region_sales_rank,
        RANK() OVER (PARTITION BY segment ORDER BY total_sales DESC) AS segment_sales_rank,
        RANK() OVER (ORDER BY total_sales DESC) AS overall_sales_rank
    FROM {catalog}.{mart_schema}.mart_customer_360
),
region_summary AS (
    SELECT
        region,
        COUNT(DISTINCT customer_id) AS total_customers,
        SUM(total_sales) AS region_sales,
        SUM(total_profit) AS region_profit,
        ROUND(AVG(avg_order_value), 2) AS avg_order_value,
        ROUND(AVG(discount_sensitivity), 2) AS avg_discount_sensitivity
    FROM {catalog}.{mart_schema}.mart_customer_360
    GROUP BY region
),
segment_summary AS (
    SELECT
        segment,
        COUNT(customer_id) AS total_customers,
        SUM(total_sales) AS segment_sales,
        SUM(total_profit) AS segment_profit,
        AVG(recency_days) AS avg_recency,
        AVG(customer_lifetime_days) AS avg_lifetime
    FROM {catalog}.{mart_schema}.mart_customer_360
    GROUP BY segment
)
SELECT
    rc.customer_id,
    rc.customer_name,
    rc.segment,
    rc.region,
    rc.total_sales,
    rc.total_profit,
    rc.total_orders,
    rc.avg_order_value,
    rc.recency_days,
    rc.customer_lifetime_days,
    rc.discount_sensitivity,
    rc.profitability_flag,
    rc.overall_sales_rank,
    rc.region_sales_rank,
    rc.segment_sales_rank,
    rs.region_sales,
    rs.region_profit,
    rs.total_customers AS region_total_customers,
    ss.segment_sales,
    ss.segment_profit,
    ss.total_customers AS segment_total_customers,
    ss.avg_recency,
    ss.avg_lifetime
FROM ranked_customers rc
LEFT JOIN region_summary rs ON rc.region = rs.region
LEFT JOIN segment_summary ss ON rc.segment = ss.segment
""")